In [1]:
import os
import shutil
import requests
import zipfile
from time import strftime
from concurrent.futures import ThreadPoolExecutor, as_completed
import random
import math
import gc

import tensorflow as tf
from tensorflow.keras import Model, Sequential
from tensorflow.keras.layers import Input, Layer, Conv2D, MaxPooling2D, Flatten, Dense, InputLayer, BatchNormalization, Dropout, ReLU
from tensorflow.keras.layers import (
    RandomFlip, RandomRotation, RandomZoom, RandomBrightness,
    RandomContrast, RandomTranslation, GaussianNoise, Reshape, Resizing, Rescaling
)
from tensorflow.keras.regularizers import L2
from tensorflow.keras.initializers import RandomNormal
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import BinaryAccuracy, FalseNegatives, FalsePositives, TruePositives, TrueNegatives, Precision, Recall, AUC
from tensorflow.keras.callbacks import Callback, ModelCheckpoint, ReduceLROnPlateau, LearningRateScheduler
from tensorflow.keras.saving import register_keras_serializable
from tensorflow.keras.models import load_model
from tensorflow.random import normal
from tensorflow.keras import backend as K


import matplotlib.pyplot as plt
import heapq

2025-04-22 18:46:14.709138: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745347574.922161      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745347574.978920      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
# BASE_PATH = os.path.dirname(__file__)
BASE_PATH = os.getcwd()

config = {
    "data_folder": os.path.join(BASE_PATH, "data"),
    "POS_PATH": os.path.join(BASE_PATH, 'data' , 'positive'),
    "NEG_PATH": os.path.join(BASE_PATH, 'data', 'negative'),
    "ANC_PATH": os.path.join(BASE_PATH, 'data', 'anchor'),
    "VAL_PATH": os.path.join(BASE_PATH, 'data', 'validation'),

    "dataset_url": "https://www.kaggle.com/api/v1/datasets/download/jessicali9530/lfw-dataset",
    "dataset_url2": "https://www.kaggle.com/api/v1/datasets/download/hearfool/vggface2",
    "save_model_folder": os.path.join(BASE_PATH, "saved_models"),

    "IM_SIZE": 100,
    "REGULARIZATION_RATE": 0.01,
    "DROPOUT_RATE_CONV": 0.3,
    "DROPOUT_RATE_DENSE": 0.5,
    "LEARNING_RATE": 0.0001,
    "EPOCHS": 5,
    "BATCH_SIZE": 32,
    "TRAINING_RATIO": 0.8,
    "VALIDATION_RATIO": 0.1,
    "TESTING_RATIO": 0.1,
    "GENERATOR_ITER": 1,
    "DigiFace_gen_no": 1
}


# os.makedirs(config["save_model_folder"], exist_ok=True)
os.makedirs(config["data_folder"], exist_ok=True)
os.makedirs(config["POS_PATH"], exist_ok=True)
os.makedirs(config["NEG_PATH"], exist_ok=True)
os.makedirs(config["ANC_PATH"], exist_ok=True)
os.makedirs(config["VAL_PATH"], exist_ok=True)

In [3]:
!git clone https://github.com/zllrunning/face-parsing.PyTorch.git

Cloning into 'face-parsing.PyTorch'...
remote: Enumerating objects: 92, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 92 (delta 21), reused 17 (delta 17), pack-reused 63 (from 1)
Receiving objects: 100% (92/92), 3.06 MiB | 25.49 MiB/s, done.
Resolving deltas: 100% (23/23), done.


In [4]:
!ls
!mv ./face-parsing.PyTorch/model.py ./model.py
!mv ./face-parsing.PyTorch/resnet.py ./resnet.py
!ls
!rm -fdr ./face-parsing.PyTorch

data  face-parsing.PyTorch  __notebook__.ipynb
data  face-parsing.PyTorch  model.py  __notebook__.ipynb  resnet.py


In [5]:
def download_file(name, url, target_dir):
    zip_path = os.path.join(target_dir, f"{name}.zip")
    path = os.path.join(target_dir, name)

    try:
        print(f"Starting download for {name} from {url}...")

        with requests.get(url, stream=True) as r:
            r.raise_for_status()
            with open(zip_path, 'wb') as f_out:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f_out.write(chunk)

            # Extract the ZIP file
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(path)

        # Remove the ZIP file after extraction
        os.remove(zip_path)

        print(f"[✓] Finished downloading {name}")
        return path

    except requests.exceptions.RequestException as e:
        print(f"[✗] Failed to download {name}: {e}")
        return None

In [6]:
def download_data(dataset_url, target_dir):
    os.makedirs(target_dir, exist_ok=True)

    zip_path = os.path.join(target_dir, "data.zip")

    path = os.path.join(target_dir, 'lfw')
    path_image_dirs = os.path.join(target_dir, 'lfw/lfw-deepfunneled/lfw-deepfunneled/')

    download_file("lfw", dataset_url, target_dir)

    POS_PATH = config["POS_PATH"]
    NEG_PATH = config["NEG_PATH"]
    ANC_PATH = config["ANC_PATH"]
    VAL_PATH = config["VAL_PATH"]

    if not os.path.exists(POS_PATH):
        os.makedirs(POS_PATH)
    if not os.path.exists(NEG_PATH):
        os.makedirs(NEG_PATH)
    if not os.path.exists(ANC_PATH):
        os.makedirs(ANC_PATH)
    if not os.path.exists(VAL_PATH):
        os.makedirs(VAL_PATH)

    for directory in os.listdir(path_image_dirs):
        for file in os.listdir(os.path.join(path_image_dirs, directory)):
            EX_PATH = os.path.join(path_image_dirs, directory, file)
            NEW_PATH = os.path.join(NEG_PATH, file)
            os.replace(EX_PATH, NEW_PATH)

    try:
        shutil.rmtree(path)
        print(f"Directory '{path}' and its contents removed successfully.")
    except FileNotFoundError:
        print(f"Directory '{path}' not found.")
    except OSError as e:
        print(f"Error: {e}")
    print("Download data done!!")

In [7]:
def download_data2(dataset_url, target_dir):
    zip_path = os.path.join(target_dir, "data2.zip")
    path = os.path.join(target_dir, 'VGGFace2')

    download_file("VGGFace2", dataset_url, target_dir)

    POS_PATH = config["POS_PATH"]             # to 1
    NEG_PATH = config["NEG_PATH"]
    ANC_PATH = config["ANC_PATH"]             # to 2

    trainPath = os.path.join(path, 'train')   # from
    valPath = os.path.join(path, 'val')

    for dir in [trainPath, valPath]:
        items = os.listdir(dir)
        for i in items:
            p = os.path.join(dir, i)
            pos_i = os.path.join(POS_PATH, i)
            anc_i = os.path.join(ANC_PATH, i)
            os.makedirs(pos_i, exist_ok=True)
            os.makedirs(anc_i, exist_ok=True)

            img  = os.listdir(p)

            half = len(img) // 2
            first_half = img[:half]
            second_half = img[half:]

            # Move first half
            for item in first_half:
                shutil.move(os.path.join(p, item), os.path.join(pos_i, item))

            # Move second half
            for item in second_half:
                shutil.move(os.path.join(p, item), os.path.join(anc_i, item))

    try:
        shutil.rmtree(path)
        print(f"Directory '{path}' and its contents removed successfully.")
    except FileNotFoundError:
        print(f"Directory '{path}' not found.")
    except OSError as e:
        print(f"Error: {e}")
    print("Download data done!!")

In [8]:
download_data(config["dataset_url"], config["data_folder"])
download_data2(config["dataset_url2"], config["data_folder"])

Starting download for lfw from https://www.kaggle.com/api/v1/datasets/download/jessicali9530/lfw-dataset...
[✓] Finished downloading lfw
Directory '/kaggle/working/data/lfw' and its contents removed successfully.
Download data done!!
Starting download for VGGFace2 from https://www.kaggle.com/api/v1/datasets/download/hearfool/vggface2...
[✓] Finished downloading VGGFace2
Directory '/kaggle/working/data/VGGFace2' and its contents removed successfully.
Download data done!!


In [9]:
!pip install gdown
!gdown --id 154JgKpzCPW82qINcVieuPH3fZ2e0P812 -O ./79999_iter.pth

/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=154JgKpzCPW82qINcVieuPH3fZ2e0P812
To: /kaggle/working/79999_iter.pth
100%|███████████████████████████████████████| 53.3M/53.3M [00:00<00:00, 234MB/s]


In [10]:
!pip install gdown
!gdown --id 154JgKpzCPW82qINcVieuPH3fZ2e0P812 -O ./79999_iter.pth

/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=154JgKpzCPW82qINcVieuPH3fZ2e0P812
To: /kaggle/working/79999_iter.pth
100%|███████████████████████████████████████| 53.3M/53.3M [00:00<00:00, 164MB/s]


In [11]:
import torch
import torchvision.transforms as transforms
import numpy as np
import cv2
from PIL import Image
from model import BiSeNet
import os
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import threading

def gather_image_paths_recursive(root_folder, extensions=('.jpg', '.jpeg', '.png')):
    image_paths = []
    for dirpath, _, filenames in os.walk(root_folder):
        for f in filenames:
            if f.lower().endswith(extensions):
                image_paths.append(os.path.join(dirpath, f))
    return image_paths

n_classes = 19

def load_model():
    model = BiSeNet(n_classes=n_classes)
    model.load_state_dict(torch.load('./79999_iter.pth'))
    model.eval().cuda()
    return model

# Load 4 models once
model_pool = [load_model() for _ in range(10)]
model_lock = threading.Lock()

from itertools import cycle

model_cycle = cycle(model_pool)  # infinitely cycle through 4 models
model_assign_lock = threading.Lock()

def extract_face_region(parsing, image):
    face_parts = list(range(1, 14))
    face_mask = np.isin(parsing, face_parts).astype(np.uint8) * 255
    contours, _ = cv2.findContours(face_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        cnt = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(cnt)
        face_crop = image[y:y+h, x:x+w]
        return face_crop, (x, y, w, h)
    return None, None

def fit_face_in_canvas(face_crop, canvas_size=(256, 256)):
    canvas = np.zeros((canvas_size[1], canvas_size[0], 3), dtype=np.uint8)
    face_resized = cv2.resize(face_crop, (256, 256))
    x_offset = (canvas_size[0] - 256) // 2
    y_offset = (canvas_size[1] - 256) // 2
    canvas[y_offset:y_offset+256, x_offset:x_offset+256] = face_resized
    return canvas


def get_model_for_thread():
    with model_assign_lock:
        return next(model_cycle)

def get_parsing_with_model(image_path, model):
    image = Image.open(image_path).convert("RGB")
    transform = transforms.Compose([
        transforms.Resize((512, 512)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ])
    img_resized = image.resize((512, 512))
    with torch.no_grad():
        tensor = transform(img_resized).unsqueeze(0).cuda()
        out = model(tensor)[0]
        parsing = out.squeeze(0).cpu().numpy().argmax(0)
    return parsing, np.array(img_resized)

def process_image_with_model(image_path):
    try:
        model = get_model_for_thread()
        parsing, image_np = get_parsing_with_model(image_path, model)
        face_crop, _ = extract_face_region(parsing, image_np)
        if face_crop is not None:
            final_image = fit_face_in_canvas(face_crop)
            final_image_bgr = cv2.cvtColor(final_image, cv2.COLOR_RGB2BGR)  # Convert to BGR before saving
            cv2.imwrite(image_path, final_image_bgr)
    except Exception as e:
        print(f"Failed {image_path}: {e}")

def process_folder_recursive_multithread(root_folders, max_workers=4):
    image_paths = []
    for i in root_folders:
        image_paths.extend(gather_image_paths_recursive(i))

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        list(tqdm(executor.map(process_image_with_model, image_paths),
                  total=len(image_paths),
                  desc="Processing Images with multiple models"))

input_root_folder = [config["NEG_PATH"], config["POS_PATH"], config["ANC_PATH"]]
process_folder_recursive_multithread(input_root_folder, max_workers=10)

Downloading: "https://download.pytorch.org/models/resnet18-5c106cde.pth" to /root/.cache/torch/hub/checkpoints/resnet18-5c106cde.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 244MB/s]
/tmp/ipykernel_19/3290604043.py:24: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loade

In [12]:
!zip -rq ./data.zip ./data/

In [13]:
!rm -rfd ./data
!rm ./model.py ./resnet.py ./*.pth